# v22 Event Chunk Cache Verification

Run this before training. It checks that event chunk parquet files are readable, samples liquid and illiquid tickers, verifies quote/trade count sanity, validates cached target columns, confirms target horizons are time-based, and checks the training data provider's idle-chunk expansion.

In [ ]:
from pathlib import Path

# Edit these only if your workstation layout changes.
CACHE_ROOT = Path(r"D:\market-data\flatfiles\us_stocks_sip\derived\event_chunks_v1")
CHUNK_MS = 500
MAX_QUOTE_EVENTS = 128
MAX_TRADE_EVENTS = 192
MAX_TOTAL_EVENTS = 256

# Keep this small for a quick smoke check; set to None to scan every parquet file.
CORRUPTION_SCAN_LIMIT = 500

# Number of liquid and illiquid ticker-month files to inspect in detail.
SAMPLE_PER_SIDE = 5

# Horizon chunks to validate. Defaults match v22 target_cache_horizon_chunks.
TARGET_HORIZON_CHUNKS = (20, 40, 60, 120, 240, 600)

PARAM_ROOT = CACHE_ROOT / f"chunk_ms={CHUNK_MS}" / f"mq={MAX_QUOTE_EVENTS}_mt={MAX_TRADE_EVENTS}_m={MAX_TOTAL_EVENTS}"
MANIFEST_PATH = CACHE_ROOT / "preprocess_event_chunks_manifest.jsonl"

print(f"PARAM_ROOT={PARAM_ROOT}")
print(f"MANIFEST_PATH={MANIFEST_PATH}")

In [ ]:
import json
import math
import random
import sys
from datetime import datetime

import numpy as np
import polars as pl

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "research" / "inhouse_transformer" / "v22").exists():
            return path
    return start

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.inhouse_transformer.v22.config import DataConfig
from research.inhouse_transformer.v22.data import CHUNK_SUMMARY_COLUMNS, ticker_arrays

assert PARAM_ROOT.exists(), f"Missing chunk parameter folder: {PARAM_ROOT}"
print(f"repo_root={REPO_ROOT}")
print(f"polars={pl.__version__}")

In [ ]:
files = sorted(PARAM_ROOT.glob("ticker=*/*.parquet"))
assert files, f"No parquet files found under {PARAM_ROOT}"

file_rows = []
for path in files:
    stat = path.stat()
    file_rows.append({
        "path": str(path),
        "ticker": path.parent.name.split("=", 1)[1],
        "year_month": path.stem,
        "bytes": stat.st_size,
        "modified": datetime.fromtimestamp(stat.st_mtime),
    })

files_df = pl.DataFrame(file_rows).sort("bytes", descending=True)
print(f"files={len(files):,}")
print(f"total_gb={files_df['bytes'].sum() / 1e9:.2f}")
display(files_df.select("ticker", "year_month", "bytes", "modified").head(10))
display(files_df.select("ticker", "year_month", "bytes", "modified").tail(10))

In [ ]:
if MANIFEST_PATH.exists():
    status_counts = {}
    phase_counts = {}
    failures = []
    manifest_rows = 0
    with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            manifest_rows += 1
            status_counts[row.get("status", "<missing>")] = status_counts.get(row.get("status", "<missing>"), 0) + 1
            phase_counts[row.get("phase", "<missing>")] = phase_counts.get(row.get("phase", "<missing>"), 0) + 1
            if row.get("status") == "failed":
                failures.append(row)
    print(f"manifest_rows={manifest_rows:,}")
    print(f"phase_counts={phase_counts}")
    print(f"status_counts={status_counts}")
    if failures:
        print(f"FAILED rows={len(failures):,}")
        display(pl.DataFrame(failures[:20]))
    else:
        print("No failed rows in manifest.")
else:
    print("Manifest not found. Continuing with parquet-level checks only.")

In [ ]:
def parquet_row_count(path: Path) -> int:
    return int(pl.scan_parquet(str(path)).select(pl.len()).collect().item())

scan_paths = files if CORRUPTION_SCAN_LIMIT is None else files[: min(len(files), CORRUPTION_SCAN_LIMIT)]
bad = []
checked = 0
for path in scan_paths:
    try:
        schema = pl.scan_parquet(str(path)).collect_schema()
        row_count = parquet_row_count(path)
        required = {"ticker", "session_date", "chunk_start_ns", "chunk_end_ns", "quote_values", "trade_values", "event_kinds", "event_indices"}
        missing = sorted(required - set(schema.names()))
        if missing or row_count <= 0:
            bad.append({"path": str(path), "rows": row_count, "missing": missing, "error": "schema_or_empty"})
    except Exception as exc:
        bad.append({"path": str(path), "rows": None, "missing": None, "error": repr(exc)})
    checked += 1

print(f"parquet_files_checked={checked:,}")
if bad:
    display(pl.DataFrame(bad))
    raise RuntimeError(f"Found {len(bad)} unreadable or invalid parquet files.")
print("Parquet readability/schema smoke check passed.")

In [ ]:
top = files_df.head(SAMPLE_PER_SIDE)
bottom = files_df.filter(pl.col("bytes") > 0).tail(SAMPLE_PER_SIDE)
sample_df = pl.concat([top, bottom], how="vertical_relaxed").unique(["path"]).sort("bytes", descending=True)
sample_paths = [Path(value) for value in sample_df["path"].to_list()]
display(sample_df.select("ticker", "year_month", "bytes", "modified"))

In [ ]:
def summarize_chunk_file(path: Path) -> dict:
    df = pl.read_parquet(path)
    target_cols = [name for name in df.columns if name.startswith("target_")]
    null_exprs = [(pl.col(name).null_count() / pl.len()).alias(f"{name}_null_frac") for name in target_cols]
    base = df.select([
        pl.len().alias("rows"),
        pl.col("session_date").n_unique().alias("sessions"),
        pl.col("quote_count").sum().alias("quote_count_sum"),
        pl.col("trade_count").sum().alias("trade_count_sum"),
        pl.col("event_count").sum().alias("event_count_sum"),
        (pl.col("has_quote") > 0).sum().alias("chunks_with_quote"),
        (pl.col("has_trade") > 0).sum().alias("chunks_with_trade"),
        pl.col("latest_mid").min().alias("min_latest_mid"),
        pl.col("latest_mid").max().alias("max_latest_mid"),
    ]).to_dicts()[0]
    nulls = df.select(null_exprs).to_dicts()[0] if null_exprs else {}
    return {
        "ticker": path.parent.name.split("=", 1)[1],
        "year_month": path.stem,
        "bytes": path.stat().st_size,
        "target_columns": len(target_cols),
        **base,
        **{k: round(float(v), 6) for k, v in nulls.items()},
    }

summary = pl.DataFrame([summarize_chunk_file(path) for path in sample_paths])
display(summary)

In [ ]:
def validate_time_based_targets(path: Path, horizons: tuple[int, ...]) -> pl.DataFrame:
    df = pl.read_parquet(path).sort(["session_date", "chunk_start_ns"])
    chunk_ns = CHUNK_MS * 1_000_000
    bounds = df.group_by("session_date").agg(
        pl.col("chunk_start_ns").min().alias("min_chunk_start_ns"),
        pl.col("chunk_start_ns").max().alias("max_chunk_start_ns"),
    )
    quote_grid = (
        bounds.with_columns(
            pl.int_ranges(
                pl.col("min_chunk_start_ns"),
                pl.col("max_chunk_start_ns") + chunk_ns,
                chunk_ns,
            ).alias("chunk_start_ns")
        )
        .explode("chunk_start_ns")
        .select(["session_date", "chunk_start_ns"])
        .join(df.select(["session_date", "chunk_start_ns", "latest_bid", "latest_ask"]), on=["session_date", "chunk_start_ns"], how="left")
        .sort(["session_date", "chunk_start_ns"])
        .with_columns(
            pl.col("latest_bid").forward_fill().over("session_date"),
            pl.col("latest_ask").forward_fill().over("session_date"),
        )
    )
    rows = []
    for horizon in horizons:
        required = [f"target_bid_h{horizon}", f"target_ask_h{horizon}", f"target_mid_h{horizon}"]
        if any(name not in df.columns for name in required):
            rows.append({"horizon_chunks": horizon, "status": "missing_columns"})
            continue
        expected = quote_grid.with_columns(
            pl.col("latest_bid").shift(-horizon).over("session_date").alias("expected_bid"),
            pl.col("latest_ask").shift(-horizon).over("session_date").alias("expected_ask"),
        ).with_columns(((pl.col("expected_bid") + pl.col("expected_ask")) * 0.5).alias("expected_mid"))
        check = df.select(["session_date", "chunk_start_ns", *required]).join(
            expected.select(["session_date", "chunk_start_ns", "expected_bid", "expected_ask", "expected_mid"]),
            on=["session_date", "chunk_start_ns"],
            how="left",
        )
        stats = check.select([
            pl.len().alias("rows"),
            pl.col(required[2]).null_count().alias("target_mid_nulls"),
            (pl.col(required[0]) - pl.col("expected_bid")).abs().max().alias("max_bid_abs_diff"),
            (pl.col(required[1]) - pl.col("expected_ask")).abs().max().alias("max_ask_abs_diff"),
            (pl.col(required[2]) - pl.col("expected_mid")).abs().max().alias("max_mid_abs_diff"),
        ]).to_dicts()[0]
        rows.append({"horizon_chunks": horizon, "horizon_seconds": horizon * CHUNK_MS / 1000.0, "status": "ok", **stats})
    return pl.DataFrame(rows)

target_checks = []
for path in sample_paths:
    check = validate_time_based_targets(path, TARGET_HORIZON_CHUNKS).with_columns(
        pl.lit(path.parent.name.split("=", 1)[1]).alias("ticker"),
        pl.lit(path.stem).alias("year_month"),
    )
    target_checks.append(check)
target_check_df = pl.concat(target_checks, how="vertical_relaxed")
display(target_check_df.select("ticker", "year_month", "horizon_chunks", "horizon_seconds", "status", "rows", "target_mid_nulls", "max_bid_abs_diff", "max_ask_abs_diff", "max_mid_abs_diff"))

bad_targets = target_check_df.filter(
    (pl.col("status") != "ok") |
    (pl.col("max_bid_abs_diff").fill_null(0.0) > 1e-6) |
    (pl.col("max_ask_abs_diff").fill_null(0.0) > 1e-6) |
    (pl.col("max_mid_abs_diff").fill_null(0.0) > 1e-6)
)
if bad_targets.height:
    display(bad_targets)
    raise RuntimeError("Cached target columns do not match dense time-grid shifts.")
print("Time-based target validation passed for sampled files.")

In [ ]:
def inspect_idle_expansion(path: Path) -> dict:
    config = DataConfig(
        cache_root=CACHE_ROOT,
        chunk_ms=CHUNK_MS,
        max_quote_events=MAX_QUOTE_EVENTS,
        max_trade_events=MAX_TRADE_EVENTS,
        max_total_events=MAX_TOTAL_EVENTS,
    )
    frame = pl.read_parquet(path).sort(["session_date", "chunk_start_ns"])
    first_session = frame.get_column("session_date")[0]
    one_session = frame.filter(pl.col("session_date") == first_session)
    arrays = ticker_arrays(one_session, config)
    if arrays is None:
        return {"ticker": path.parent.name.split("=", 1)[1], "year_month": path.stem, "session": first_session, "status": "no_arrays"}
    event_count_idx = CHUNK_SUMMARY_COLUMNS.index("event_count")
    seconds_since_quote_idx = CHUNK_SUMMARY_COLUMNS.index("seconds_since_quote")
    seconds_since_trade_idx = CHUNK_SUMMARY_COLUMNS.index("seconds_since_trade")
    event_counts = arrays["chunk_summary"][:, event_count_idx]
    return {
        "ticker": path.parent.name.split("=", 1)[1],
        "year_month": path.stem,
        "session": first_session,
        "sparse_rows_session": one_session.height,
        "dense_rows_session": int(arrays["chunk_summary"].shape[0]),
        "idle_rows_session": int((event_counts == 0.0).sum()),
        "max_seconds_since_quote": float(arrays["chunk_summary"][:, seconds_since_quote_idx].max()),
        "max_seconds_since_trade": float(arrays["chunk_summary"][:, seconds_since_trade_idx].max()),
        "status": "ok",
    }

idle_rows = [inspect_idle_expansion(path) for path in sample_paths]
idle_df = pl.DataFrame(idle_rows)
display(idle_df)

if idle_df.filter((pl.col("status") == "ok") & (pl.col("dense_rows_session") < pl.col("sparse_rows_session"))).height:
    raise RuntimeError("Dense idle expansion produced fewer rows than sparse input for at least one sample.")
print("Idle expansion check passed for sampled files.")

In [ ]:

# Semantic raw-vs-chunk verification for one ticker and one origin timestamp.
# This intentionally compares raw normalized quote/trade rows to the materialized sparse event chunks.
# If this cell fails, inspect the printed tables before rebuilding/training.

from datetime import datetime, timezone
from zoneinfo import ZoneInfo

from research.inhouse_transformer.v22.data import (
    QUOTE_FEATURE_COLUMNS,
    TRADE_FEATURE_COLUMNS,
    attach_quote_state_to_trades,
    chunk_counts,
    prepare_quote_events_for_chunks,
    prepare_trade_events_for_chunks,
    scan_normalized_quotes,
    scan_normalized_trades,
    selected_event_order_chunks,
    selected_quote_chunk_values,
    selected_trade_chunk_values,
)

SEMANTIC_TICKER = "AAPL"
SEMANTIC_ORIGIN_UTC = "2025-12-01T15:30:00Z"
SEMANTIC_FLATFILES_ROOT = CACHE_ROOT.parent.parent
SEMANTIC_MAX_DISPLAY_ROWS = 8
SEMANTIC_COMPARE_TOLERANCE = 1e-3

semantic_config = DataConfig(
    flatfiles_root=SEMANTIC_FLATFILES_ROOT,
    cache_root=CACHE_ROOT,
    chunk_ms=CHUNK_MS,
    max_quote_events=MAX_QUOTE_EVENTS,
    max_trade_events=MAX_TRADE_EVENTS,
    max_total_events=MAX_TOTAL_EVENTS,
    target_cache_horizon_chunks=TARGET_HORIZON_CHUNKS,
)

chunk_ns = CHUNK_MS * 1_000_000
origin_dt = datetime.fromisoformat(SEMANTIC_ORIGIN_UTC.replace("Z", "+00:00")).astimezone(timezone.utc)
origin_ns = int(origin_dt.timestamp() * 1_000_000_000)
origin_chunk_start_ns = (origin_ns // chunk_ns) * chunk_ns
market_dt = origin_dt.astimezone(ZoneInfo(semantic_config.session_timezone))
semantic_session = market_dt.date().isoformat()
semantic_year_month = semantic_session[:7]
context_chunks = semantic_config.context_chunks
context_start_ns = origin_chunk_start_ns - (context_chunks - 1) * chunk_ns
context_end_ns = origin_chunk_start_ns + chunk_ns
max_horizon = max(TARGET_HORIZON_CHUNKS)
target_end_ns = origin_chunk_start_ns + (max_horizon + 1) * chunk_ns
chunk_path = PARAM_ROOT / f"ticker={SEMANTIC_TICKER}" / f"{semantic_year_month}.parquet"

print(f"semantic_ticker={SEMANTIC_TICKER}")
print(f"origin_utc={origin_dt.isoformat()} origin_chunk_start_ns={origin_chunk_start_ns}")
print(f"market_session={semantic_session} market_time={market_dt.isoformat()}")
print(f"raw_flatfiles_root={SEMANTIC_FLATFILES_ROOT}")
print(f"chunk_path={chunk_path}")
assert chunk_path.exists(), f"Missing chunk parquet: {chunk_path}"

print("Reading normalized raw quotes/trades for the session. This scans the raw csv.gz files once per side.")
raw_quotes = scan_normalized_quotes(semantic_config, semantic_session, (SEMANTIC_TICKER,)).collect().sort(["sip_timestamp", "sequence_number"])
raw_trades = scan_normalized_trades(semantic_config, semantic_session, (SEMANTIC_TICKER,)).collect().sort(["sip_timestamp", "sequence_number"])
raw_quotes_window = raw_quotes.filter((pl.col("sip_timestamp") >= context_start_ns) & (pl.col("sip_timestamp") < target_end_ns))
raw_trades_window = raw_trades.filter((pl.col("sip_timestamp") >= context_start_ns) & (pl.col("sip_timestamp") < target_end_ns))
raw_quotes_context = raw_quotes.filter((pl.col("sip_timestamp") >= context_start_ns) & (pl.col("sip_timestamp") < context_end_ns))
raw_trades_context = raw_trades.filter((pl.col("sip_timestamp") >= context_start_ns) & (pl.col("sip_timestamp") < context_end_ns))

print(
    f"raw_session quotes={raw_quotes.height:,} trades={raw_trades.height:,} | "
    f"raw_context quotes={raw_quotes_context.height:,} trades={raw_trades_context.height:,} | "
    f"raw_context_plus_horizons quotes={raw_quotes_window.height:,} trades={raw_trades_window.height:,}"
)
display(raw_quotes_window.select(["ticker", "sip_timestamp", "sequence_number", "bid_price", "ask_price", "mid_price"]).head(SEMANTIC_MAX_DISPLAY_ROWS))
display(raw_trades_window.select(["ticker", "sip_timestamp", "sequence_number", "price", "size", "exchange"]).head(SEMANTIC_MAX_DISPLAY_ROWS))

cached = (
    pl.read_parquet(chunk_path)
    .filter(
        (pl.col("session_date") == semantic_session)
        & (pl.col("chunk_start_ns") >= context_start_ns)
        & (pl.col("chunk_start_ns") < target_end_ns)
    )
    .sort("chunk_start_ns")
)
assert not cached.is_empty(), "No cached chunk rows found around the semantic verification timestamp."

cached_context = cached.filter((pl.col("chunk_start_ns") >= context_start_ns) & (pl.col("chunk_start_ns") < context_end_ns))
print(
    f"cached_context_sparse_rows={cached_context.height:,} expected_context_chunks={context_chunks:,} "
    f"cached_context_quote_count_sum={cached_context['quote_count'].sum() if cached_context.height else 0:,} "
    f"cached_context_trade_count_sum={cached_context['trade_count'].sum() if cached_context.height else 0:,}"
)

trades_with_quote = attach_quote_state_to_trades(raw_trades_context, raw_quotes)
quotes_prepared = prepare_quote_events_for_chunks(raw_quotes_context, chunk_ns)
trades_prepared = prepare_trade_events_for_chunks(trades_with_quote, chunk_ns)
expected_counts = chunk_counts(semantic_config, quotes_prepared, trades_prepared).sort("chunk_start_ns")
expected_quote_values = selected_quote_chunk_values(semantic_config, quotes_prepared, expected_counts, chunk_ns).sort("chunk_start_ns")
expected_trade_values = selected_trade_chunk_values(semantic_config, trades_prepared, expected_counts, chunk_ns).sort("chunk_start_ns")
expected_order = selected_event_order_chunks(semantic_config, quotes_prepared, trades_prepared, expected_counts).sort("chunk_start_ns")

expected_context = (
    expected_counts.join(expected_quote_values, on=["session_date", "chunk_start_ns"], how="left")
    .join(expected_trade_values, on=["session_date", "chunk_start_ns"], how="left")
    .join(expected_order, on=["session_date", "chunk_start_ns"], how="left")
    .sort("chunk_start_ns")
)

expected_chunks = set(expected_context["chunk_start_ns"].to_list()) if expected_context.height else set()
cached_chunks = set(cached_context["chunk_start_ns"].to_list()) if cached_context.height else set()
missing_cached = sorted(expected_chunks - cached_chunks)
extra_cached = sorted(cached_chunks - expected_chunks)
print(f"expected_nonempty_context_chunks={len(expected_chunks):,} cached_nonempty_context_chunks={len(cached_chunks):,}")
assert not missing_cached, f"Cached chunk file is missing chunks that exist in raw context: {missing_cached[:10]}"
assert not extra_cached, f"Cached chunk file has chunks with no raw context events: {extra_cached[:10]}"

expected_by_chunk = {row["chunk_start_ns"]: row for row in expected_context.to_dicts()}
cached_by_chunk = {row["chunk_start_ns"]: row for row in cached_context.to_dicts()}


def nested_float_array(value, width: int) -> np.ndarray:
    if value is None:
        return np.zeros((0, width), dtype=np.float32)
    arr = np.asarray(value, dtype=np.float32)
    if arr.size == 0:
        return np.zeros((0, width), dtype=np.float32)
    return arr.reshape(-1, width)


def int_array(value) -> np.ndarray:
    if value is None:
        return np.zeros((0,), dtype=np.int64)
    arr = np.asarray(value, dtype=np.int64)
    return arr.reshape(-1)

comparison_rows = []
for chunk_start in sorted(expected_chunks):
    expected_row = expected_by_chunk[chunk_start]
    cached_row = cached_by_chunk[chunk_start]
    expected_quote = nested_float_array(expected_row.get("quote_values"), len(QUOTE_FEATURE_COLUMNS))
    cached_quote = nested_float_array(cached_row.get("quote_values"), len(QUOTE_FEATURE_COLUMNS))
    expected_trade = nested_float_array(expected_row.get("trade_values"), len(TRADE_FEATURE_COLUMNS))
    cached_trade = nested_float_array(cached_row.get("trade_values"), len(TRADE_FEATURE_COLUMNS))
    expected_kinds = int_array(expected_row.get("event_kinds"))
    cached_kinds = int_array(cached_row.get("event_kinds"))
    expected_indices = int_array(expected_row.get("event_indices"))
    cached_indices = int_array(cached_row.get("event_indices"))

    quote_diff = 0.0 if expected_quote.shape == cached_quote.shape and expected_quote.size == 0 else (
        float(np.max(np.abs(expected_quote - cached_quote))) if expected_quote.shape == cached_quote.shape else math.inf
    )
    trade_diff = 0.0 if expected_trade.shape == cached_trade.shape and expected_trade.size == 0 else (
        float(np.max(np.abs(expected_trade - cached_trade))) if expected_trade.shape == cached_trade.shape else math.inf
    )
    order_match = bool(np.array_equal(expected_kinds, cached_kinds) and np.array_equal(expected_indices, cached_indices))
    comparison_rows.append(
        {
            "chunk_start_ns": chunk_start,
            "raw_quote_count": float(expected_row.get("quote_count") or 0.0),
            "cached_quote_count": float(cached_row.get("quote_count") or 0.0),
            "raw_trade_count": float(expected_row.get("trade_count") or 0.0),
            "cached_trade_count": float(cached_row.get("trade_count") or 0.0),
            "selected_quote_rows": int(cached_quote.shape[0]),
            "selected_trade_rows": int(cached_trade.shape[0]),
            "quote_values_max_abs_diff": quote_diff,
            "trade_values_max_abs_diff": trade_diff,
            "event_order_match": order_match,
        }
    )

comparison_df = pl.DataFrame(comparison_rows)
display(comparison_df.head(SEMANTIC_MAX_DISPLAY_ROWS))

bad_comparisons = comparison_df.filter(
    (pl.col("raw_quote_count") != pl.col("cached_quote_count"))
    | (pl.col("raw_trade_count") != pl.col("cached_trade_count"))
    | (pl.col("quote_values_max_abs_diff") > SEMANTIC_COMPARE_TOLERANCE)
    | (pl.col("trade_values_max_abs_diff") > SEMANTIC_COMPARE_TOLERANCE)
    | (~pl.col("event_order_match"))
)
if bad_comparisons.height:
    display(bad_comparisons.head(20))
assert bad_comparisons.is_empty(), "Raw context events do not match cached event chunk context."

origin_row = cached.filter(pl.col("chunk_start_ns") == origin_chunk_start_ns)
assert origin_row.height == 1, f"Expected exactly one sparse origin chunk row at {origin_chunk_start_ns}, got {origin_row.height}. Pick a timestamp inside a non-idle chunk."
origin = origin_row.row(0, named=True)

target_rows = []
for horizon in TARGET_HORIZON_CHUNKS:
    target_chunk_start = origin_chunk_start_ns + int(horizon) * chunk_ns
    # Cached target semantics: latest quote state after all valid quote updates inside the target chunk,
    # forward-filled from prior chunks if that target chunk has no quote update.
    raw_target_quote = raw_quotes.filter(pl.col("sip_timestamp") < (target_chunk_start + chunk_ns)).tail(1)
    expected_bid = float(raw_target_quote["bid_price"][0]) if raw_target_quote.height else None
    expected_ask = float(raw_target_quote["ask_price"][0]) if raw_target_quote.height else None
    expected_mid = ((expected_bid + expected_ask) * 0.5) if expected_bid is not None and expected_ask is not None else None
    cached_bid = origin.get(f"target_bid_h{horizon}")
    cached_ask = origin.get(f"target_ask_h{horizon}")
    cached_mid = origin.get(f"target_mid_h{horizon}")
    target_rows.append(
        {
            "horizon_chunks": int(horizon),
            "horizon_seconds": float(horizon * CHUNK_MS / 1000.0),
            "target_chunk_start_ns": target_chunk_start,
            "expected_bid": expected_bid,
            "cached_bid": cached_bid,
            "expected_ask": expected_ask,
            "cached_ask": cached_ask,
            "expected_mid": expected_mid,
            "cached_mid": cached_mid,
            "bid_abs_diff": None if expected_bid is None or cached_bid is None else abs(expected_bid - cached_bid),
            "ask_abs_diff": None if expected_ask is None or cached_ask is None else abs(expected_ask - cached_ask),
            "mid_abs_diff": None if expected_mid is None or cached_mid is None else abs(expected_mid - cached_mid),
        }
    )

target_df = pl.DataFrame(target_rows)
display(target_df)
bad_targets = target_df.filter(
    (pl.col("bid_abs_diff").fill_null(math.inf) > SEMANTIC_COMPARE_TOLERANCE)
    | (pl.col("ask_abs_diff").fill_null(math.inf) > SEMANTIC_COMPARE_TOLERANCE)
    | (pl.col("mid_abs_diff").fill_null(math.inf) > SEMANTIC_COMPARE_TOLERANCE)
)
assert bad_targets.is_empty(), "Cached horizon targets do not match raw quote-derived target states."

print("Semantic raw-vs-chunk verification passed.")


In [ ]:
print("Verification complete.")
print("Review the displayed tables above. If there were no exceptions, the sampled cache is ready for training.")